# AIPA-Eval 모델 파인튜닝 (LoRA)

**목표**: Claude Haiku가 생성한 학습 데이터로 Qwen 2.5-7B를 LoRA 파인튜닝 → AIPA-Eval 모델 생성

**C# 비유**: 미리 학습된 대형 클래스(Qwen)에 작은 어댑터 클래스(LoRA)를 상속시켜서, 특정 작업에 특화된 서브클래스(AIPA-Eval)를 만드는 것

**필요 환경**: Google Colab (무료 T4 GPU)

---

## 실행 순서
1. GPU 확인 & 라이브러리 설치
2. 학습 데이터 업로드 & 전처리
3. 베이스 모델 로드 (Qwen 2.5-7B)
4. LoRA 설정 & 학습
5. 테스트 추론
6. 어댑터 저장 (Google Drive)

## 1단계: GPU 확인 & 라이브러리 설치

Colab 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행

In [ ]:
# GPU 확인 (T4 GPU가 보여야 함)
!nvidia-smi

In [ ]:
# 필요 라이브러리 설치
# C#의 NuGet 패키지 설치와 동일
!pip install -q \
    torch \
    transformers>=4.44.0 \
    datasets \
    peft>=0.12.0 \
    trl>=0.9.0 \
    bitsandbytes>=0.43.0 \
    accelerate>=0.33.0 \
    sentencepiece \
    protobuf

## 2단계: 학습 데이터 업로드 & 전처리

좌측 파일 탭(📁)에서 `training_data_deduped.jsonl` 파일을 드래그 앤 드롭으로 업로드

파일 위치: `AIPA_Engine\data\processed\training_data_deduped.jsonl` (772건)

In [ ]:
# Google Drive 마운트 (학습 결과 저장용)
# C#의 FileSystem.Mount() 같은 것 - 내 Google Drive를 폴더처럼 접근 가능
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import glob

# 업로드된 학습 데이터 파일 찾기
data_files = (
    glob.glob('/content/training_data_deduped.jsonl') +
    glob.glob('/content/training_data_*.jsonl') +
    glob.glob('/content/drive/MyDrive/AIPA/training_data_*.jsonl')
)
print(f"발견된 데이터 파일: {data_files}")

# ★ 파일 못 찾으면 여기에 직접 경로 입력
# data_files = ['/content/training_data_deduped.jsonl']

In [ ]:
# JSONL → 학습용 포맷으로 변환
# LoRA 학습에 필요한 형태: {"text": "<프롬프트>\n<응답>"}
#
# C# 비유: List<TrainingExample>을 순회하면서 string 형식으로 변환하는 것

def format_training_example(item: dict) -> str:
    """
    학습 데이터 1건을 프롬프트+응답 형식으로 변환
    
    input: {stimulus, stimulus_type, persona_*, axes}
    output: {evaluations: [{axis, score, reasoning}], summary}
    """
    inp = item["input"]
    out = item["output"]
    
    # 프롬프트 조립 (모델에게 주는 지시문)
    prompt = f"""당신은 소비자 반응 평가 전문가입니다.

다음 자극물에 대해 주어진 페르소나의 관점에서 평가해주세요.

[자극물]
유형: {inp['stimulus_type']}
내용: {inp['stimulus']}

[페르소나]
연령대: {inp['persona_age_group']}
성별: {inp['persona_gender']}
직업: {inp['persona_occupation']}
소득: {inp['persona_income']}
특성: {', '.join(inp['persona_traits'])}

[평가 축]
{', '.join(inp['axes'])}

각 축에 대해 1-10점 점수와 근거를 JSON으로 응답하세요."""
    
    # 응답 (모델이 생성해야 할 정답)
    response = json.dumps(out, ensure_ascii=False)
    
    # Qwen 챗 템플릿 형식
    return f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n{response}<|im_end|>"


# 모든 데이터 로드 & 변환
training_texts = []
for fpath in data_files:
    with open(fpath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            item = json.loads(line)
            training_texts.append(format_training_example(item))

print(f"총 학습 데이터: {len(training_texts)}건")
if training_texts:
    print(f"\n=== 샘플 (첫 번째) ===")
    print(training_texts[0][:500] + "...")

In [ ]:
# Hugging Face Dataset으로 변환
# C# 비유: List<string> → IDataset 인터페이스로 래핑
from datasets import Dataset

# 90% 학습, 10% 검증 (과적합 방지)
import random
random.shuffle(training_texts)
split_idx = int(len(training_texts) * 0.9)

train_dataset = Dataset.from_dict({"text": training_texts[:split_idx]})
eval_dataset = Dataset.from_dict({"text": training_texts[split_idx:]})

print(f"학습 데이터: {len(train_dataset)}건")
print(f"검증 데이터: {len(eval_dataset)}건")

## 3단계: 베이스 모델 로드 (Qwen 2.5-7B)

Hugging Face에서 자동 다운로드 (~14GB) → Colab 서버 메모리에만 로드

4bit 양자화로 메모리 절약 (14GB → ~4GB)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 베이스 모델명 (Hugging Face 저장소)
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# 4bit 양자화 설정 (C#의 압축 옵션 같은 것)
# 14GB 모델을 ~4GB로 압축해서 T4 GPU(16GB)에 올릴 수 있게 함
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                        # 4bit 양자화 활성화
    bnb_4bit_quant_type="nf4",               # NormalFloat4 양자화 (정확도 높음)
    bnb_4bit_compute_dtype=torch.float16,     # 연산은 float16으로
    bnb_4bit_use_double_quant=True,           # 이중 양자화 (메모리 추가 절약)
)

print("토크나이저 로딩...")
# 토크나이저 = 텍스트를 숫자(토큰)로 변환하는 도구
# C# 비유: Encoding.UTF8.GetBytes()의 AI 버전
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token  # 패딩 토큰 설정

print("모델 로딩... (약 5분 소요)")
# 모델 로드 (Hugging Face에서 자동 다운로드 → GPU 메모리에 올림)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,     # 4bit 양자화 적용
    device_map="auto",                  # GPU 자동 배치
    trust_remote_code=True,
)

print(f"모델 로드 완료! GPU 메모리: {torch.cuda.memory_allocated()/1024**3:.1f}GB")

## 4단계: LoRA 설정 & 학습

LoRA = 전체 모델(7B 파라미터)을 건드리지 않고, 작은 어댑터(~8M 파라미터)만 학습

C# 비유: 거대한 부모 클래스를 수정하지 않고, 자식 클래스에서 override한 메서드만 학습

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 양자화된 모델을 학습 가능하게 준비
model = prepare_model_for_kbit_training(model)

# LoRA 하이퍼파라미터 설정
lora_config = LoraConfig(
    r=16,                          # LoRA 랭크 (높을수록 표현력↑, 메모리↑). 16이 가성비 좋음
    lora_alpha=32,                 # 스케일링 팩터 (보통 r의 2배)
    lora_dropout=0.05,             # 드롭아웃 (과적합 방지)
    bias="none",                   # 바이어스는 학습 안 함
    task_type="CAUSAL_LM",         # 텍스트 생성 모델
    target_modules=[               # LoRA를 적용할 레이어 (어텐션 레이어들)
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

# LoRA 어댑터 적용
model = get_peft_model(model, lora_config)

# 학습 가능한 파라미터 수 확인
model.print_trainable_parameters()
# 예상 출력: trainable params: 8,388,608 || all params: 7,623,004,160 || trainable%: 0.11%
# → 전체의 0.11%만 학습! 나머지는 동결(freeze)

In [ ]:
from trl import SFTTrainer, SFTConfig

# 학습 설정
training_args = SFTConfig(
    output_dir="./aipa-eval-lora",

    # 학습 하이퍼파라미터
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,

    # 메모리 최적화 — fp16 사용, bf16 명시적 비활성화
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",

    # 로깅 & 저장
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",

    # 기타
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print("학습 준비 완료!")

In [ ]:
# ★★★ 학습 시작 ★★★
# 1000건 기준 약 30~60분 소요 (T4 GPU)
# loss가 점점 줄어드는지 확인 → 잘 학습되고 있다는 의미
print("=" * 50)
print("학습 시작! (약 30~60분 소요)")
print("=" * 50)

trainer.train()

print("\n" + "=" * 50)
print("학습 완료!")
print("=" * 50)

## 5단계: 테스트 추론

학습된 모델로 실제 평가를 생성해서 품질 확인

In [ ]:
# 테스트 입력
test_prompt = """당신은 소비자 반응 평가 전문가입니다.

다음 자극물에 대해 주어진 페르소나의 관점에서 평가해주세요.

[자극물]
유형: 광고
내용: 프리미엄 유기농 샐러드 배달 서비스 - 매일 신선한 재료로 만든 맞춤형 샐러드

[페르소나]
연령대: 20대
성별: 여성
직업: IT 개발자
소득: 중상
특성: 건강관심, 바쁜일상, SNS활발

[평가 축]
구매의향, 브랜드호감도, 가격적절성, 재구매의사, 추천의향

각 축에 대해 1-10점 점수와 근거를 JSON으로 응답하세요."""

# 토큰화 & 추론
inputs = tokenizer(
    f"<|im_start|>user\n{test_prompt}<|im_end|>\n<|im_start|>assistant\n",
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7,
        do_sample=True,
    )

# 결과 디코딩
result = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=== AIPA-Eval 모델 출력 ===")
print(result)

## 6단계: 어댑터 저장

학습된 LoRA 어댑터만 저장 (~50MB)

전체 모델(14GB)이 아닌 어댑터만 저장하면 됨

In [ ]:
import os

# 1. Colab 로컬에 저장
ADAPTER_PATH = "./aipa-eval-lora-final"
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"로컬 저장 완료: {ADAPTER_PATH}")

# 2. Google Drive에 백업 (Colab 세션 종료돼도 보존됨)
DRIVE_PATH = "/content/drive/MyDrive/AIPA/aipa-eval-lora"
os.makedirs(DRIVE_PATH, exist_ok=True)
model.save_pretrained(DRIVE_PATH)
tokenizer.save_pretrained(DRIVE_PATH)
print(f"Google Drive 저장 완료: {DRIVE_PATH}")

# 저장된 파일 크기 확인
total_size = sum(os.path.getsize(os.path.join(DRIVE_PATH, f)) for f in os.listdir(DRIVE_PATH))
print(f"\n어댑터 총 크기: {total_size / 1024 / 1024:.1f}MB")

## 완료!

### 저장된 것
- Google Drive: `AIPA/aipa-eval-lora/` (~50MB)

### 다음 단계
1. Cloud Run 배포 시: 베이스 모델(Qwen) + 어댑터(LoRA)를 합쳐서 서빙
2. 학습 데이터 추가 생성 후: 이 노트북 다시 실행하면 성능 향상

### 참고
- 어댑터만 있으면 됨 (베이스 모델은 Hugging Face에서 항상 다운 가능)
- 추가 학습 시 `trainer.train(resume_from_checkpoint=True)` 사용 가능